# VAE — Variational Autoencoder

Trains on CelebA: compresses 128×128 RGB → latent `(4, 16, 16)` → reconstructs back.

**Bugs fixed vs original:**
- `Sigmoid` → `Tanh` output (images normalised to `[-1,1]`; Sigmoid outputs `[0,1]` → MSE is always wrong)
- Pre-norm `ResBlock` for stable gradient flow
- KL weight `0.001` → `1e-4` (large enough to regularise posterior, small enough to not crush reconstruction)
- `Adam` → `AdamW` + gradient clipping
- `save_image` now unnormalises `[-1,1]→[0,1]` so saved PNGs look correct
- `weights_only=True` on checkpoint load (PyTorch security best practice)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
class ResBlock(nn.Module):
    """Pre-norm residual block — identity at init, gradient-friendly."""
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(8, channels), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x):
        return x + self.net(x)


class VAE(nn.Module):
    """
    Encoder: 128->64->32->16, outputs mean+logvar (8ch)
    Decoder: 16->32->64->128, outputs Tanh in [-1,1]

    FIX: original Sigmoid output [0,1] vs normalised input [-1,1] — MSE broken.
    """
    def __init__(self, in_channels=3, latent_channels=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64,  4, stride=2, padding=1), nn.SiLU(),  # 128->64
            nn.Conv2d(64,  128, 4, stride=2, padding=1), nn.SiLU(),           # 64->32
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.SiLU(),           # 32->16
            ResBlock(256),
            nn.GroupNorm(8, 256), nn.SiLU(),
            nn.Conv2d(256, latent_channels * 2, 3, padding=1),                # mean+logvar
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 256, 3, padding=1), nn.SiLU(),
            ResBlock(256),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.SiLU(),  # 16->32
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1), nn.SiLU(),  # 32->64
            nn.ConvTranspose2d(64, in_channels, 4, stride=2, padding=1),      # 64->128
            nn.Tanh(),  # FIX: Sigmoid -> Tanh to match [-1,1] input range
        )

    def encode(self, x):
        mean, logvar = torch.chunk(self.encoder(x), 2, dim=1)
        logvar = logvar.clamp(-30.0, 20.0)
        z = mean + torch.randn_like(mean) * (0.5 * logvar).exp()
        return z, mean, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z, mean, logvar = self.encode(x)
        return self.decode(z), mean, logvar

In [ ]:
class CelebDataset(Dataset):
    def __init__(self, root_dir, image_size=128):
        self.root_dir = root_dir
        # FIX: filter to image files only (original used os.listdir with no filter)
        self.images = [f for f in os.listdir(root_dir)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # [0,1] -> [-1,1]
        ])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.images[idx])
        return self.transform(Image.open(path).convert("RGB"))


dataset    = CelebDataset("data/img_align_celeba")
subset     = Subset(dataset, range(min(5000, len(dataset))))
dataloader = DataLoader(subset, batch_size=8, shuffle=True, num_workers=0)
print(f"Dataset size: {len(subset)}, batches: {len(dataloader)}")

In [ ]:
# Shape sanity check
model = VAE().to(device)
with torch.no_grad():
    x = next(iter(dataloader)).to(device)
    recon, mean, logvar = model(x)
print(f"Input:  {x.shape}    range [{x.min():.2f}, {x.max():.2f}]")
print(f"Recon:  {recon.shape}  range [{recon.min():.2f}, {recon.max():.2f}]")
print(f"Latent: {mean.shape}")
assert x.shape == recon.shape
print("Shapes OK")

In [ ]:
KL_WEIGHT = 1e-4  # FIX: was 0.001 (too large early) — 1e-4 balances recon vs KL
EPOCHS    = 10

model     = VAE().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  # FIX: AdamW
os.makedirs("images", exist_ok=True)
os.makedirs("models", exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = total_recon = total_kl = 0.0

    bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images in bar:
        images = images.to(device)
        recon, mean, logvar = model(images)

        recon_loss = F.mse_loss(recon, images)
        kl_loss    = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
        loss       = recon_loss + KL_WEIGHT * kl_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # FIX: added
        optimizer.step()

        total_loss  += loss.item()
        total_recon += recon_loss.item()
        total_kl    += kl_loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}", recon=f"{recon_loss.item():.4f}", kl=f"{kl_loss.item():.2f}")

    n = len(dataloader)
    print(f"Epoch {epoch+1}  loss={total_loss/n:.4f}  recon={total_recon/n:.4f}  kl={total_kl/n:.2f}")

    # FIX: unnormalise [-1,1]->[0,1] before saving so PNGs look correct
    with torch.no_grad():
        model.eval()
        grid = torch.cat([images[:8], recon[:8]]).clamp(-1, 1) * 0.5 + 0.5
        save_image(grid, f"images/vae_epoch_{epoch+1}.png", nrow=8)
        model.train()

In [ ]:
torch.save(model.state_dict(), "models/vae.pth")
print("Saved -> models/vae.pth")

In [ ]:
# Reload and verify
model = VAE().to(device)
model.load_state_dict(torch.load("models/vae.pth", map_location=device, weights_only=True))
# FIX: weights_only=True avoids PyTorch security warning on newer versions
model.eval()
print("Loaded OK")

with torch.no_grad():
    images = next(iter(dataloader)).to(device)
    recon, mean, logvar = model(images)

print(f"Recon shape: {recon.shape}")
print(f"Output range: [{recon.min():.3f}, {recon.max():.3f}]  (expected: -1 to 1)")